In [1]:
import numpy as np 
import pandas as pd 
import MLM.angle_between as angle
import MLM.moire_lattice_vector_finder as mlf 
import MLM.structure_writer as sw
from matplotlib.path import Path

In [2]:
def select_atoms_in_polygon_path(df, polygon_path, tolerance):
    """
    Selects rows from a DataFrame where the 'x' and 'y' coordinates
    are inside or within a given tolerance of the specified polygon path.
    """
    xy_points = df[['x','y']].values  # Extract x, y coordinates
    # Add the radius parameter to include points near the boundary
    inside_mask = polygon_path.contains_points(xy_points, radius=tolerance)
    return df[inside_mask]

In [3]:
def write_vasp_optimized(filename, A1, A2, A3, atom_data, sort_order):
    # Buffer for all the lines to write
    element_type = " ".join(f"{key}" for key, _ in sort_order.items())
    element_count = " ".join(f"{len(sorted_df[sorted_df['type']==key])}" for key, _ in sort_order.items())
    lines = [
        "System Generated by MLM \n",
        "1.0\n",
        f"{A1[0]} {A1[1]} 0.0\n",
        f"{A2[0]} {A2[1]} 0.0\n",
        f"{A3[0]} {A3[1]} {A3[2]}\n",
        f"{element_type}\n",
        f"{element_count} \n",
        "Cartesian\n"
    ]
    atomic_positions = atom_data[['x', 'y', 'z']]
    # Write the buffer to the file
    with open(filename, 'w') as file:
        file.writelines(lines)
        # Use to_csv for efficient DataFrame writing
        atomic_positions.to_csv(file, sep=' ', index=False, header=False, mode='a')

In [4]:
file_1 = "../moire_structures/mos2/mos2_monolayer.vasp"
file_2 = "../moire_structures/mos2/mos2_monolayer.vasp"
file_3 = "../moire_structures/mos2/mos2_monolayer.vasp"

In [5]:
lattice_vectors1, atom_type_arr1, dat1 = mlf.read_vasp(file_1)

lattice_vectors2, atom_type_arr2, dat2 = mlf.read_vasp(file_2)

lattice_vectors3, atom_type_arr3, dat3 = mlf.read_vasp(file_3)

In [6]:
df = pd.read_pickle("mos2_ttl_for_generation.pkl")

In [7]:
df = df.sort_values(by='norm', ascending=True)

In [8]:
df.head(10)

,Theta,Deg_Diff,Phi,A1,A2,norm,delvec,delang,Theta_bin,Phi_bin
85,38.22,38.20,76.42,"[-3.1660028102, 21.9346889744]","[17.4129975662, 13.709180608999999]",38.385704,0.002487,0.000007,"(38.0, 39.0]",NaN
38,21.79,21.78,43.57,"[3.1659969686, 21.9346889744]","[20.578998185800003, 8.2255083654]",38.385704,0.001387,0.000007,"(21.0, 22.0]","(43.0, 44.0]"
86,38.22,43.56,81.78,"[-11.081002168600001, 19.192852852599998]","[11.080997057200001, 19.192852852599998]",38.385706,0.002629,0.000004,"(38.0, 39.0]",NaN
37,21.79,16.42,38.21,"[-11.081002168600001, 19.192852852599998]","[11.080997057200001, 19.192852852599998]",38.385706,0.001242,0.000004,"(21.0, 22.0]","(38.0, 39.0]"
54,27.80,10.41,38.21,"[-1.5830039608000002, 30.1601973398]","[25.3279969246, 16.4510167308]",52.310897,0.001695,0.000008,"(27.0, 28.0]","(38.0, 39.0]"
69,32.21,49.58,81.79,"[1.5829959286, 30.1601973398]","[26.9109972344, 13.709180608999999]",52.310897,0.001688,0.000008,"(32.0, 33.0]",NaN
65,32.21,21.78,53.99,"[1.5829959286, 30.1601973398]","[26.9109972344, 13.709180608999999]",52.310897,0.000540,0.000008,"(32.0, 33.0]","(53.0, 54.0]"
39,21.79,32.20,53.99,"[1.5829959286, 30.1601973398]","[26.9109972344, 13.709180608999999]",52.310897,0.000540,0.000008,"(21.0, 22.0]","(53.0, 54.0]"
36,21.79,10.41,32.20,"[1.5829959286, 30.1601973398]","[26.9109972344, 13.709180608999999]",52.310897,0.002231,0.000008,"(21.0, 22.0]","(32.0, 33.0]"
35,21.79,6.00,27.79,"[-12.6640032086, 27.418361217999998]","[17.4129961058, 24.6765250962]",52.310899,0.003043,0.000005,"(21.0, 22.0]","(27.0, 28.0]"


In [9]:
import warnings
warnings.filterwarnings('ignore')


count = 1
for row in df.iterrows():
    A1 = np.array(row[1]['A1'])
    A2 = np.array(row[1]['A2'])
    norm = row[1]['norm']
    replicate = int((4*norm/np.linalg.norm(lattice_vectors1['a'])))
    new_lattice_vectors1, new_total_atoms1, replicated_atom1, replicated_type_arr1 = sw.replicate_atoms(a = lattice_vectors1['a'],
                                                                                             b = lattice_vectors1['b'],
                                                                                             c = lattice_vectors1['c'],
                                                                                             atom_data = dat1,
                                                                                             atom_type_arr = atom_type_arr1,
                                                                                             natoms = len(dat1),
                                                                                             na = replicate,
                                                                                             nb = replicate,
                                                                                             nc = 1)
    new_lattice_vectors2, new_total_atoms2, replicated_atom2, replicated_type_arr2 = sw.replicate_atoms(a = lattice_vectors2['a'],
                                                                                             b = lattice_vectors2['b'],
                                                                                             c = lattice_vectors2['c'],
                                                                                             atom_data = dat2,
                                                                                             atom_type_arr = atom_type_arr2,
                                                                                             natoms = len(dat2),
                                                                                             na = replicate,
                                                                                             nb = replicate,
                                                                                             nc = 1)
    new_lattice_vectors3, new_total_atoms3, replicated_atom3, replicated_type_arr3 = sw.replicate_atoms(a = lattice_vectors3['a'],
                                                                                             b = lattice_vectors3['b'],
                                                                                             c = lattice_vectors3['c'],
                                                                                             atom_data = dat3,
                                                                                             atom_type_arr = atom_type_arr3,
                                                                                             natoms = len(dat3),
                                                                                             na = replicate,
                                                                                             nb = replicate,
                                                                                             nc = 1)
    # Create a DataFrame for positions
    positions1_df = pd.DataFrame(replicated_atom1, columns=['x', 'y', 'z'])

    # Add the atom types to the DataFrame
    positions1_df['type'] = replicated_type_arr1
    
    positions2_df = pd.DataFrame(replicated_atom2, columns=['x', 'y', 'z'])

    # Add the atom types to the DataFrame
    positions2_df['type'] = replicated_type_arr2 
    positions2_df['z'] = positions2_df['z'] + 6.5
    
    # Create a DataFrame for positions
    positions3_df = pd.DataFrame(replicated_atom3, columns=['x', 'y', 'z'])

    # Add the atom types to the DataFrame
    positions3_df['type'] = replicated_type_arr3
    positions3_df['z'] = positions3_df['z'] + 6.5*2 
    
    theta1 = row[1]['Theta']
    
    theta_rad = np.deg2rad(theta1)
    
    rotation_matrix = np.array([[np.cos(theta_rad), -np.sin(theta_rad), 0],
                            [np.sin(theta_rad), np.cos(theta_rad), 0],
                            [0, 0, 1]])
    
    middle_layer_positions = positions2_df[['x', 'y', 'z']].values  # Extract x, y, z positions
    rotated_positions = middle_layer_positions @ rotation_matrix.T  # Apply rotation

    # Create a new DataFrame for the rotated positions
    rotated_middle_layer = pd.DataFrame(rotated_positions, columns=['x', 'y', 'z'])

    # Add the atom types back to the rotated DataFrame
    rotated_middle_layer['type'] = positions2_df['type'].values
    
    theta2 = row[1]['Phi']
    theta_rad = np.deg2rad(theta2)
    
    rotation_matrix = np.array([[np.cos(theta_rad), -np.sin(theta_rad), 0],
                            [np.sin(theta_rad), np.cos(theta_rad), 0],
                            [0, 0, 1]])



    # Rotate the positions in the upper layer
    upper_layer_positions = positions3_df[['x', 'y', 'z']].values  # Extract x, y, z positions
    rotated_positions = upper_layer_positions @ rotation_matrix.T  # Apply rotation

    # Create a new DataFrame for the rotated positions
    rotated_upper_layer = pd.DataFrame(rotated_positions, columns=['x', 'y', 'z'])

    # Add the atom types back to the rotated DataFrame
    rotated_upper_layer['type'] = positions3_df['type'].values
    
    # Concatenate DataFrames vertically
    concat_vertical = pd.concat([positions1_df, rotated_middle_layer, rotated_upper_layer], ignore_index=True)
    

    A3 = np.array([0, 0, 50.0])
    
    vertex = [ [0.0,0.0], [A1[0], A1[1]],  [(A1+A2)[0],(A1+A2)[1]], [A2[0],A2[1]]]
    
    polygon_points = vertex
    
    polygon_path = Path(polygon_points)
    
    selected_atoms_df = select_atoms_in_polygon_path(concat_vertical, polygon_path, tolerance=-0.01)
    
    unique_types = selected_atoms_df['type'].unique()
    
    sort_order = {value: index for index, value in enumerate(unique_types)}
    
    selected_atoms_df['sort_key'] = selected_atoms_df['type'].map(sort_order)
    
    sorted_df = selected_atoms_df.sort_values(by='sort_key').drop(columns='sort_key').copy()
    
    write_vasp_optimized(f"test_strs_mos2/mos2_ttl_{theta1}_{theta2}_{count}.vasp",A1,A2,A3, sorted_df, sort_order)
    print(f"Done with {count} | {theta1}, {theta2}")
    count = count + 1
    
    
    


Done with 1 | 38.22, 76.4199999999926
Done with 2 | 21.79, 43.57000000000325
Done with 3 | 38.22, 81.77999999999153
Done with 4 | 21.79, 38.21000000000241
Done with 5 | 27.8, 38.21000000000147
Done with 6 | 32.21, 81.78999999999033
Done with 7 | 32.21, 53.98999999999587
Done with 8 | 21.79, 53.990000000004876
Done with 9 | 21.79, 32.20000000000147
Done with 10 | 21.79, 27.79000000000078
Done with 11 | 27.8, 49.58000000000325
Done with 12 | 32.2, 38.209999999999006
Done with 13 | 38.22, 92.19999999998946
Done with 14 | 38.22, 70.41999999999379
Done with 15 | 32.2, 70.4199999999926
Done with 16 | 46.82, 85.0399999999926
Done with 17 | 46.82, 98.20999999998998
Done with 18 | 38.22, 46.829999999998485
Done with 19 | 21.79, 73.17000000000787
Done with 20 | 13.17, 21.789999999999836
Done with 21 | 13.17, 34.95999999999956
Done with 22 | 38.22, 51.38999999999758
Done with 23 | 13.18, 38.20999999999949
Done with 24 | 13.18, 51.389999999999205
Done with 25 | 46.83, 68.60999999999586
Done with 2

: 

In [23]:
import warnings
warnings.filterwarnings('ignore')


count = 1
for row in df.iterrows():
    if count == 72:
        A1 = np.array(row[1]['A1'])
        A2 = np.array(row[1]['A2'])
        norm = row[1]['norm']
        replicate = int((4*norm/np.linalg.norm(lattice_vectors1['a'])))
        new_lattice_vectors1, new_total_atoms1, replicated_atom1, replicated_type_arr1 = sw.replicate_atoms(a = lattice_vectors1['a'],
                                                                                                    b = lattice_vectors1['b'],
                                                                                                    c = lattice_vectors1['c'],
                                                                                                    atom_data = dat1,
                                                                                                    atom_type_arr = atom_type_arr1,
                                                                                                    natoms = len(dat1),
                                                                                                    na = replicate,
                                                                                                    nb = replicate,
                                                                                                    nc = 1)
        new_lattice_vectors2, new_total_atoms2, replicated_atom2, replicated_type_arr2 = sw.replicate_atoms(a = lattice_vectors2['a'],
                                                                                                    b = lattice_vectors2['b'],
                                                                                                    c = lattice_vectors2['c'],
                                                                                                    atom_data = dat2,
                                                                                                    atom_type_arr = atom_type_arr2,
                                                                                                    natoms = len(dat2),
                                                                                                    na = replicate,
                                                                                                    nb = replicate,
                                                                                                    nc = 1)
        new_lattice_vectors3, new_total_atoms3, replicated_atom3, replicated_type_arr3 = sw.replicate_atoms(a = lattice_vectors3['a'],
                                                                                                    b = lattice_vectors3['b'],
                                                                                                    c = lattice_vectors3['c'],
                                                                                                    atom_data = dat3,
                                                                                                    atom_type_arr = atom_type_arr3,
                                                                                                    natoms = len(dat3),
                                                                                                    na = replicate,
                                                                                                    nb = replicate,
                                                                                                    nc = 1)
        # Create a DataFrame for positions
        positions1_df = pd.DataFrame(replicated_atom1, columns=['x', 'y', 'z'])

        # Add the atom types to the DataFrame
        positions1_df['type'] = replicated_type_arr1

        positions2_df = pd.DataFrame(replicated_atom2, columns=['x', 'y', 'z'])

        # Add the atom types to the DataFrame
        positions2_df['type'] = replicated_type_arr2 
        positions2_df['z'] = positions2_df['z'] + 6.5

        # Create a DataFrame for positions
        positions3_df = pd.DataFrame(replicated_atom3, columns=['x', 'y', 'z'])

        # Add the atom types to the DataFrame
        positions3_df['type'] = replicated_type_arr3
        positions3_df['z'] = positions3_df['z'] + 6.5*2 

        theta1 = row[1]['Theta']

        theta_rad = np.deg2rad(theta1)

        rotation_matrix = np.array([[np.cos(theta_rad), -np.sin(theta_rad), 0],
                                [np.sin(theta_rad), np.cos(theta_rad), 0],
                                [0, 0, 1]])

        middle_layer_positions = positions2_df[['x', 'y', 'z']].values  # Extract x, y, z positions
        rotated_positions = middle_layer_positions @ rotation_matrix.T  # Apply rotation

        # Create a new DataFrame for the rotated positions
        rotated_middle_layer = pd.DataFrame(rotated_positions, columns=['x', 'y', 'z'])

        # Add the atom types back to the rotated DataFrame
        rotated_middle_layer['type'] = positions2_df['type'].values

        theta2 = np.round(row[1]['Phi'],2)
        theta_rad = np.deg2rad(theta2)

        rotation_matrix = np.array([[np.cos(theta_rad), -np.sin(theta_rad), 0],
                                [np.sin(theta_rad), np.cos(theta_rad), 0],
                                [0, 0, 1]])



        # Rotate the positions in the upper layer
        upper_layer_positions = positions3_df[['x', 'y', 'z']].values  # Extract x, y, z positions
        rotated_positions = upper_layer_positions @ rotation_matrix.T  # Apply rotation

        # Create a new DataFrame for the rotated positions
        rotated_upper_layer = pd.DataFrame(rotated_positions, columns=['x', 'y', 'z'])

        # Add the atom types back to the rotated DataFrame
        rotated_upper_layer['type'] = positions3_df['type'].values

        # Concatenate DataFrames vertically
        concat_vertical = pd.concat([positions1_df, rotated_middle_layer, rotated_upper_layer], ignore_index=True)


        A3 = np.array([0, 0, 50.0])

        vertex = [ [0.0,0.0], [A1[0], A1[1]],  [(A1+A2)[0],(A1+A2)[1]], [A2[0],A2[1]]]

        polygon_points = vertex

        polygon_path = Path(polygon_points)

        selected_atoms_df = select_atoms_in_polygon_path(concat_vertical, polygon_path, tolerance=-0.01)

        unique_types = selected_atoms_df['type'].unique()

        sort_order = {value: index for index, value in enumerate(unique_types)}

        selected_atoms_df['sort_key'] = selected_atoms_df['type'].map(sort_order)

        sorted_df = selected_atoms_df.sort_values(by='sort_key').drop(columns='sort_key').copy()

        write_vasp_optimized(f"test_strs_mos2/mos2_ttl_{theta1}_{theta2}_{count}.vasp",A1,A2,A3, sorted_df, sort_order)
        print(f"Done with {count} | {theta1}, {theta2}")

    count = count + 1
    
    
    


Done with 72 | 50.57, 101.14
